**2. Install the packages for scCL**

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os

Mounted at /content/drive


In [2]:
!pwd

/content


In [3]:
!pip install torch torchvision torchaudio
!pip install pytorch-lightning
!pip install anndata
!pip install lightning-bolts
!pip install scanpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 141.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.8/300.8 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.5/829.5 kB 42.2 MB/s eta 0:00:00
  Attempting uninstall: pytorch-lightning
    Found existing installation: pytorch-lightning 2.6.1
    Uninstalling pytorch-lightning-2.6.1:
      Successfully uninstalled pytorch-lightning-2.6.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 41.8 MB/s eta 0:00:00


In [4]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
import anndata as ad
from sklearn.preprocessing import LabelEncoder
import random

In [5]:
# -------------------------
# 1. Load train/test data
# -------------------------
# Using the same data directories from your CE script
adata_train = ad.read_h5ad("/content/drive/MyDrive/Colab Notebooks/data/Biddy_train.h5ad")
adata_test  = ad.read_h5ad("/content/drive/MyDrive/Colab Notebooks/data/Biddy_test.h5ad")

X_train = adata_train.X.toarray() if hasattr(adata_train.X, "toarray") else adata_train.X
X_test  = adata_test.X.toarray() if hasattr(adata_test.X, "toarray") else adata_test.X

train_raw_labels = adata_train.obs["clone_id"].values
test_raw_labels  = adata_test.obs["clone_id"].values

# -------------------------
# 2. Encode labels
# -------------------------
le = LabelEncoder()
y_train = le.fit_transform(train_raw_labels)
y_test = le.transform(test_raw_labels)


In [6]:
# -------------------------
# 3. Custom Triplet Dataset
# -------------------------
class TripletDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

        # Pre-compute indices for each class to sample positives/negatives quickly
        self.labels = self.y.numpy()
        self.classes = np.unique(self.labels)
        self.class_to_indices = {
            cls: np.where(self.labels == cls)[0] for cls in self.classes
        }

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        # Anchor
        anchor = self.X[idx]
        label = self.labels[idx]

        # Positive (sample from same class)
        pos_indices = self.class_to_indices[label]
        # If a lineage only has 1 cell, it will sample itself (distance = 0)
        pos_idx = np.random.choice(pos_indices)
        positive = self.X[pos_idx]

        # Negative (sample from any other class)
        neg_classes = self.classes[self.classes != label]
        neg_class = np.random.choice(neg_classes)
        neg_idx = np.random.choice(self.class_to_indices[neg_class])
        negative = self.X[neg_idx]

        return anchor, positive, negative

train_dataset = TripletDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

In [7]:

# -------------------------
# 4. Define Triplet Model
# -------------------------
class TripletEmbeddingModel(nn.Module):
    def __init__(self, input_dim, hidden_dims=(1024, 256), embedding_dim=32):
        super().__init__()

        # Identical base encoder to your CE model, but no classification head
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dims[0]),
            nn.BatchNorm1d(hidden_dims[0]),
            nn.ReLU(),
            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.BatchNorm1d(hidden_dims[1]),
            nn.ReLU(),
            nn.Linear(hidden_dims[1], embedding_dim)
            # Removed final ReLU to allow full hypersphere projection
        )

    def forward(self, x):
        emb = self.encoder(x)
        # L2 Normalize the embeddings (Standard for distance-based Triplet Loss)
        emb = F.normalize(emb, p=2, dim=1)
        return emb

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

model = TripletEmbeddingModel(input_dim=X_train.shape[1]).to(device)

# Standard Triplet Margin Loss (margin=1.0 is a good default for L2 normalized vectors)
criterion = nn.TripletMarginLoss(margin=1.0, p=2)
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-6)



In [8]:
# -------------------------
# 5. Train
# -------------------------
epochs = 50
model.train()

for epoch in range(epochs):
    total_loss = 0.0

    for anchor, positive, negative in train_loader:
        anchor = anchor.to(device)
        positive = positive.to(device)
        negative = negative.to(device)

        optimizer.zero_grad()

        # Forward pass all three through the model
        emb_a = model(anchor)
        emb_p = model(positive)
        emb_n = model(negative)

        # Calculate loss
        loss = criterion(emb_a, emb_p, emb_n)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss / len(train_loader):.4f}")



Epoch 10/50 | Loss: 0.1183
Epoch 20/50 | Loss: 0.0711
Epoch 30/50 | Loss: 0.0379
Epoch 40/50 | Loss: 0.0575
Epoch 50/50 | Loss: 0.0333


In [9]:
# -------------------------
# 6. Extract embeddings
# -------------------------
def extract_embeddings(X_array, batch_size=512):
    dataset = TensorDataset(torch.tensor(X_array, dtype=torch.float32))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    all_embeddings = []
    model.eval()
    with torch.no_grad():
        for (batch_X,) in loader:
            batch_X = batch_X.to(device)
            emb = model(batch_X)
            all_embeddings.append(emb.cpu().numpy())

    return np.concatenate(all_embeddings, axis=0)

train_embeddings = extract_embeddings(X_train)
test_embeddings = extract_embeddings(X_test)

print("Train embeddings:", train_embeddings.shape)
print("Test embeddings:", test_embeddings.shape)

# -------------------------
# 7. Save for existing eval scripts
# -------------------------
# Saved with "_Triplet_" notation to distinguish from your CE runs
np.save("/content/drive/MyDrive/Colab Notebooks/other_method/Biddy_Triplet_train_embeddings.npy", train_embeddings)
np.save("/content/drive/MyDrive/Colab Notebooks/other_method/Biddy_Triplet_test_embeddings.npy", test_embeddings)

Train embeddings: (5893, 32)
Test embeddings: (641, 32)
